# Public Research Workflow Demo

This notebook demonstrates a **non-proprietary** version of the WaveFilter research workflow using synthetic data only.

It intentionally does **not** contain the production indicator formula, calibrated parameters, Pine Script, AutoLab implementation, or trading rules.

The goal is to illustrate four research practices:

1. causal event construction without look-ahead,
2. time-ordered development / holdout splitting,
3. block-bootstrap uncertainty analysis,
4. separating a volatility-risk finding from directional trading claims.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')


## 1. Generate synthetic market regimes

The synthetic series alternates between quieter and more volatile regimes. The construction is deliberately simple and is not intended to imitate the private production model.


In [ ]:
n = 6000
index = pd.date_range('2025-01-01', periods=n, freq='h')

# Synthetic volatility regimes: quiet -> active -> quiet -> active.
regime = np.zeros(n)
regime[1200:2200] = 1
regime[3500:4700] = 1

sigma = np.where(regime == 1, 0.012, 0.004)
returns = rng.normal(0.0, sigma)

# Add a small number of synthetic shocks.
shock_idx = rng.choice(np.arange(200, n - 200), size=45, replace=False)
returns[shock_idx] += rng.normal(0.0, 0.028, size=len(shock_idx))

price = 100 * np.exp(np.cumsum(returns))

df = pd.DataFrame({
    'return': returns,
    'price': price,
    'abs_return': np.abs(returns),
    'synthetic_regime': regime,
}, index=index)

df.head()


## 2. Build a causal illustrative volatility event

Every threshold below is computed from information available **before** the current bar by using `.shift(1)`. The values are illustrative and are not production parameters.


In [ ]:
history = 168

# Historical reference levels use only prior observations.
prior_low_vol = (
    df['abs_return']
    .rolling(history, min_periods=history)
    .quantile(0.35)
    .shift(1)
)

prior_high_move = (
    df['abs_return']
    .rolling(history, min_periods=history)
    .quantile(0.85)
    .shift(1)
)

prior_short_vol = df['abs_return'].rolling(24, min_periods=24).mean().shift(1)

# Generic demonstration event: a large current move after a relatively quiet background.
df['demo_event'] = (
    (prior_short_vol <= prior_low_vol * 1.35) &
    (df['abs_return'] >= prior_high_move)
)

# A future 6-bar volatility label. This is an evaluation target, not an input.
future_abs = pd.concat(
    [df['abs_return'].shift(-i) for i in range(1, 7)],
    axis=1,
)
df['future_6bar_mean_abs_return'] = future_abs.mean(axis=1)

# Causal reference threshold known at event time.
future_vol_threshold = (
    df['abs_return']
    .rolling(history, min_periods=history)
    .quantile(0.70)
    .shift(1)
)
df['future_high_vol'] = df['future_6bar_mean_abs_return'] > future_vol_threshold

df[['price', 'demo_event', 'future_high_vol']].dropna().head()


## 3. Time-ordered development and sealed-style holdout split

The final 25% of observations are kept separate from development. No random shuffle is used.


In [ ]:
usable = df.dropna(subset=['future_high_vol']).copy()
cut = int(len(usable) * 0.75)

development = usable.iloc[:cut].copy()
holdout = usable.iloc[cut:].copy()

print('Development:', development.index.min(), 'to', development.index.max(), len(development))
print('Holdout:    ', holdout.index.min(), 'to', holdout.index.max(), len(holdout))


## 4. Evaluate lift without making a directional trading claim

The question is simply whether the event identifies a higher probability of future volatility. It does **not** ask whether price will rise or fall.


In [ ]:
def volatility_lift(frame: pd.DataFrame) -> pd.Series:
    base = frame['future_high_vol'].mean()
    events = frame.loc[frame['demo_event'], 'future_high_vol']
    event_rate = events.mean() if len(events) else np.nan
    return pd.Series({
        'bars': len(frame),
        'events': int(frame['demo_event'].sum()),
        'baseline_high_vol_rate': base,
        'event_high_vol_rate': event_rate,
        'lift': event_rate - base,
    })

summary = pd.DataFrame({
    'development': volatility_lift(development),
    'holdout': volatility_lift(holdout),
}).T

summary


## 5. Block-bootstrap uncertainty

Time-series events are serially correlated, so an ordinary independent-sample bootstrap can understate uncertainty. This example resamples contiguous time blocks.


In [ ]:
def block_bootstrap_lift(frame, block_size=72, n_boot=2000, seed=7):
    local_rng = np.random.default_rng(seed)
    n = len(frame)
    values = []

    if n < block_size * 2:
        raise ValueError('Not enough rows for the requested block size.')

    starts = np.arange(0, n - block_size + 1)
    blocks_needed = int(np.ceil(n / block_size))

    for _ in range(n_boot):
        sampled = []
        for start in local_rng.choice(starts, size=blocks_needed, replace=True):
            sampled.append(frame.iloc[start:start + block_size])
        boot = pd.concat(sampled, axis=0).iloc[:n]

        base = boot['future_high_vol'].mean()
        events = boot.loc[boot['demo_event'], 'future_high_vol']
        if len(events) == 0:
            continue
        values.append(events.mean() - base)

    return np.asarray(values)

boot = block_bootstrap_lift(holdout)

pd.Series({
    'bootstrap_mean_lift': boot.mean(),
    'p05': np.quantile(boot, 0.05),
    'median': np.quantile(boot, 0.50),
    'p95': np.quantile(boot, 0.95),
    'P(lift > 0)': np.mean(boot > 0),
})


## 6. Visualize synthetic events

The chart below is only a synthetic demonstration of how a public research notebook can communicate event timing without exposing the production indicator.


In [ ]:
view = holdout.iloc[:700].copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(view.index, view['price'], linewidth=1.5, label='Synthetic price')

event_rows = view[view['demo_event']]
ax.scatter(
    event_rows.index,
    event_rows['price'],
    marker='o',
    s=30,
    label='Illustrative volatility event',
)

ax.set_title('Synthetic causal event demonstration')
ax.set_ylabel('Synthetic price')
ax.legend()
plt.tight_layout()
plt.show()


## Takeaway

This notebook demonstrates the **research process**, not the private indicator:

- event features are causal,
- evaluation targets may use future data only after the event is fixed,
- time splits remain ordered,
- holdout performance is reported separately,
- uncertainty is estimated with time blocks,
- volatility findings are not automatically converted into directional trading signals.

The production implementation remains private.
